In [1]:
import pyarrow.feather as feather
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from item_classifier import *
from plots import *

NameError: name 'ItemClassifier' is not defined

In [ ]:
table = feather.read_table('../dataset/data_andre_fulfilled.feather', memory_map=True) 
df= table.to_pandas() 
df.head()

In [ ]:
df_indexed = df.copy()
df_indexed["date"] = pd.to_datetime(df_indexed["date"])
df_indexed = df_indexed.set_index("date").sort_index()
df_indexed.head()

In [ ]:
'''
# Classify all items
df_classified, summary = classify_items(
    df_indexed,
    seasonal_strength_threshold=0.3,    # Adjust based on your domain
    cv_threshold=0.5,                    # Adjust for intermittence detection
    initial_gap_fraction=0.2,            # 20% gap at start = new item
)

# Display summary
print("Label Distribution:")
print(summary['label_counts'])
print("\n" + "="*60)

df_classified.head()
'''



## Classification Parameters Guide

The classifier uses three main parameters to categorize items:

### 1. `seasonal_strength_threshold` (default: 0.3)
- **What**: Strength of periodicity/seasonality in sales pattern (0-1 scale)
- **Higher values**: More strict seasonality detection
- **Lower values**: Easier to detect seasonal patterns
- **From document**: Items with sales/non-sales cycles that are regular and periodic

### 2. `cv_threshold` (default: 0.5)  
- **What**: Coefficient of Variation in cycle intervals
- **Higher values**: Allow more variability in intervals (more "erratic")
- **Lower values**: Stricter intermittence detection
- **From document**: Intermittent items have intervals that "don't share any common pattern"

### 3. `initial_gap_fraction` (default: 0.2)
- **What**: Fraction of zero-sales at the beginning (0-1 scale)
- **Higher values**: Allow longer initial gaps
- **Lower values**: Stricter new item detection
- **From document**: New items have "meaningful non-sales cycle at the beginning"

## Tuning Tips
- Start with defaults, then adjust based on your visualization results
- If too many items are "seasonal" → increase `seasonal_strength_threshold`
- If too many items are "intermittent" → increase `cv_threshold`
- If too many items are "new" → increase `initial_gap_fraction`


In [ ]:
#plot_label_distribution(summary["label_counts"]);

In [ ]:
df_classified, labels_df, summary = classify_items_adi_cv(df_indexed)


In [ ]:
df_indexed.head()

In [ ]:
df_classified

In [ ]:
plot_label_distribution(summary['label_counts']);

In [ ]:
plot_product_value_evolution(df_indexed, selected_product=151825, date_col=None, value_col='value')


In [ ]:
plot_random_sample_per_label(df_indexed, df_classified, date_col=None, seed=None);

In [ ]:
df_classified[(df_classified['item_label']=='seasonal') | (df_classified['item_label']=='smooth')]

In [ ]:
def select_best_erratic_smooth(df, top_n=100, max_zero_rate=0.2, sales_col='value', item_col='item_id', label_col='item_label', labels=('erratic','smooth'), by='total_sales'):
    # filter erratic / smooth
    df_sub = df[df[label_col].isin(labels)].copy()

    # ensure numeric sales
    df_sub[sales_col] = pd.to_numeric(df_sub[sales_col], errors='coerce').fillna(0)

    # compute metrics per item
    agg = df_sub.groupby(item_col).agg(
        days_observed=('date', 'nunique'),
        total_sales=(sales_col, 'sum'),
        mean_sales=(sales_col, 'mean'),
        median_sales=(sales_col, 'median'),
        zeros=(sales_col, lambda x: (x==0).sum()),
        obs_count=(sales_col, 'count')
    ).reset_index()

    agg['zero_rate'] = agg['zeros'] / agg['obs_count']
    agg['nonzero_mean'] = agg.apply(lambda r: r['total_sales'] / max(1, r['obs_count'] - r['zeros']), axis=1)

    # filter by zero rate
    agg = agg[agg['zero_rate'] <= max_zero_rate]

    # choose sort key
    if by == 'total_sales':
        agg = agg.sort_values('total_sales', ascending=False)
    elif by == 'mean_sales':
        agg = agg.sort_values('mean_sales', ascending=False)
    elif by == 'nonzero_mean':
        agg = agg.sort_values('nonzero_mean', ascending=False)
    else:
        raise ValueError("by must be 'total_sales', 'mean_sales' or 'nonzero_mean'")

    # take top_n items and return original rows for those items
    top_items = agg.head(top_n)[item_col].tolist()
    return df_sub[df_sub[item_col].isin(top_items)].copy(), agg.loc[agg[item_col].isin(top_items)]

# Example usage:
# subset_rows, top_summary = select_best_erratic_smooth(df_classified, top_n=50, max_zero_rate=0.15)


In [ ]:
# Show a few example products for each ADI-CV category
n_examples = 5
for cat in ["smooth", "erratic", "intermittent", "lumpy"]:
    items_in_cat = labels_df.loc[labels_df["label"] == cat].sort_values("total_periods", ascending=False)
    print(f"\n=== {cat.upper()} ({len(items_in_cat)} items) ===")
    display(items_in_cat.head(n_examples))


In [ ]:
# Export item_id -> ADI/CV label (smooth / seasonal / intermittent / lumpy) to CSV
df_labels_adi_cv = (
    labels_df[["item_id", "label"]]
    .drop_duplicates(subset="item_id")
    .sort_values("item_id")
    .reset_index(drop=True)
)

out_path = "../dataset/df_labels_adi_cv.csv"
df_labels_adi_cv.to_csv(out_path, index=False)
print(f"Saved {len(df_labels_adi_cv)} rows -> {out_path}")
print(df_labels_adi_cv["label"].value_counts())
df_labels_adi_cv.head()


In [ ]:
 
# ── Export smooth-only products to feather ───────────────────────────────
smooth_ids = labels_df.loc[labels_df["label"] == "smooth", "item_id"].unique()
 
# Filter the full indexed dataframe to smooth items only, reset index so
# 'date' becomes a regular column again (feather doesn't support DatetimeIndex)
df_smooth = (
    df_indexed[df_indexed["item_id"].isin(smooth_ids)]
    .reset_index()          # moves 'date' back to a column
)
 
out_feather = "../dataset/data_smooth.feather"
df_smooth.to_feather(out_feather)
 
print(f"Smooth products : {len(smooth_ids)}")
print(f"Rows saved      : {len(df_smooth)}")
print(f"Saved to        : {out_feather}")
df_smooth.head()
 

In [ ]:

# ── Export smooth + erratic products to feather ───────────────────────────
smooth_erratic_ids = labels_df.loc[
    labels_df["label"].isin(["smooth", "erratic"]), "item_id"
].unique()

df_smooth_erratic = (
    df_indexed[df_indexed["item_id"].isin(smooth_erratic_ids)]
    .reset_index()
)

out_feather_se = "../dataset/data_smooth_erratic.feather"
df_smooth_erratic.to_feather(out_feather_se)

print(f"Smooth + Erratic products : {len(smooth_erratic_ids)}")
print(f"Rows saved                : {len(df_smooth_erratic)}")
print(f"Saved to                  : {out_feather_se}")
df_smooth_erratic.head()


In [ ]:
se_labels = labels_df[labels_df["label"].isin(["smooth", "erratic"])].copy()

In [ ]:
# ── Sales distribution for smooth + erratic products ─────────────────────
se_labels = labels_df[labels_df["label"].isin(["smooth", "erratic"])].copy()

# Compute total & mean sales per item from the full time series
sales_agg = (
    df_indexed[df_indexed["item_id"].isin(se_labels["item_id"])]
    .groupby("item_id")["value"]
    .agg(total_sales="sum", mean_sales="mean", median_sales="median",
         nonzero_days=lambda x: (x > 0).sum())
    .reset_index()
)

se_stats = se_labels[["item_id", "label"]].merge(sales_agg, on="item_id")

# ── Filter Thresholds ─────────────────────────────────────────────────────
# Adjust these thresholds to exclude products above or below certain values
MIN_TOTAL_SALES = 0
MAX_TOTAL_SALES = float('inf')  # e.g., set to 10000 to exclude high sellers
MIN_MEAN_SALES = 0
MAX_MEAN_SALES = float('inf')   # e.g., set to 50 to exclude high mean sales

se_stats_filtered = se_stats[
    (se_stats["total_sales"] >= MIN_TOTAL_SALES) & 
    (se_stats["total_sales"] <= MAX_TOTAL_SALES) &
    (se_stats["mean_sales"] >= MIN_MEAN_SALES) &
    (se_stats["mean_sales"] <= MAX_MEAN_SALES)
].copy()

print(f"Original product count: {len(se_stats)}")
print(f"Filtered product count: {len(se_stats_filtered)}")
print(f"Excluded products: {len(se_stats) - len(se_stats_filtered)}\n")



# ── Top-selling products table (Filtered) ─────────────────────────────────
print("\nTop 30 products by total sales (after filtering):")
display(
    se_stats_filtered.sort_values("total_sales", ascending=False)
    .head(30)
    .reset_index(drop=True)
    .style.background_gradient(subset=["total_sales", "mean_sales"], cmap="YlOrRd")
)

In [ ]:
# ── Total Sales Ranked by Product ─────────────────────────────────────────

# Sort by total sales descending
ranked_stats = se_stats_filtered.sort_values("total_sales", ascending=False).reset_index(drop=True)

plt.figure(figsize=(18, 6))

# Plot the total sales as a bar chart
# If there are many products, we can just show the top N or use a line plot. 
# Here we'll show up to the top 100 as a bar chart, or all if there are fewer.
top_n = len(ranked_stats)

plt.bar(
    range(top_n), 
    ranked_stats.head(top_n)["total_sales"], 
    color='skyblue', 
    edgecolor='gray'
)

# Formatting the X axis
plt.title("Total Sales Ranked by Product (Smooth + Erratic)")
plt.xlabel("Product Rank (ordered by Total Sales)")
plt.ylabel("Total Sales")

# Add some grid lines
plt.grid(axis='y', linestyle='--', alpha=0.7)

# To view product_id on x-axis (works well if < 50 products, otherwise gets messy)
if top_n <= 50:
    plt.xticks(range(top_n), ranked_stats.head(top_n)["item_id"].astype(str), rotation=90)
else:
    # If too many products, we just label every Nth product or hide the labels to avoid overlapping
    plt.xticks(range(0, top_n, max(1, top_n//20)), 
               ranked_stats.head(top_n).iloc[::max(1, top_n//20)]["item_id"].astype(str), 
               rotation=90)

plt.xlim(-1, top_n)
plt.tight_layout()
plt.show()

In [ ]:
# ── Total Sales Ranked by Product ─────────────────────────────────────────

# Sort by mean sales descending
ranked_stats = se_stats_filtered.sort_values("mean_sales", ascending=False).reset_index(drop=True)

plt.figure(figsize=(18, 6))

# Plot the mean sales as a bar chart
# If there are many products, we can just show the top N or use a line plot. 
# Here we'll show up to the top 100 as a bar chart, or all if there are fewer.
top_n = len(ranked_stats)

plt.bar(
    range(top_n), 
    ranked_stats.head(top_n)["mean_sales"], 
    color='skyblue', 
    edgecolor='gray'
)

# Formatting the X axis
plt.title("Mean Sales Ranked by Product (Smooth + Erratic)")
plt.xlabel("Product Rank (ordered by Mean Sales)")
plt.ylabel("Mean Sales")

# Add some grid lines
plt.grid(axis='y', linestyle='--', alpha=0.7)

# To view product_id on x-axis (works well if < 50 products, otherwise gets messy)
if top_n <= 50:
    plt.xticks(range(top_n), ranked_stats.head(top_n)["item_id"].astype(str), rotation=90)
else:
    # If too many products, we just label every Nth product or hide the labels to avoid overlapping
    plt.xticks(range(0, top_n, max(1, top_n//20)), 
               ranked_stats.head(top_n).iloc[::max(1, top_n//20)]["item_id"].astype(str), 
               rotation=90)

plt.xlim(-1, top_n)
plt.tight_layout()
plt.show()

In [ ]:
feather.write_feather(df_classified, f"../dataset/data_andre_classified.feather")

In [ ]:
total_sales_threshold = 12500
mean_sales_threshold = 20

# Filter the previously computed se_stats based on these thresholds
top_products_total = se_stats[se_stats["total_sales"] >= total_sales_threshold].sort_values("total_sales", ascending=False)
top_products_mean = se_stats[se_stats["mean_sales"] >=  mean_sales_threshold].sort_values("mean_sales", ascending=False)

print(f"Products above {total_sales_threshold} total sales: {len(top_products_total)}")
display(
    top_products_total.head(40).reset_index(drop=True)
    .style.background_gradient(subset=["total_sales"], cmap="Blues")
    .set_caption("Top 30 products by Total Sales (above threshold)")
)

print(f"\nProducts above {mean_sales_threshold} mean sales: {len(top_products_mean)}")
display(
    top_products_mean.head(40).reset_index(drop=True)
    .style.background_gradient(subset=["mean_sales"], cmap="Greens")
    .set_caption("Top 30 products by Mean Sales (above threshold)")
)

In [ ]:
df_top_products_total = df_indexed[df_indexed["item_id"].isin(top_products_total["item_id"])]
feather.write_feather(df_top_products_total, f"../dataset/top_{total_sales_threshold}.feather")

In [ ]:
# ── Top Products Overall (Regardless of Label) ───────────────────────────────

# Compute total & mean sales per item for all products in the dataset
sales_agg_all = (
    df_indexed.groupby("item_id")["value"]
    .agg(total_sales="sum", mean_sales="mean", median_sales="median",
         nonzero_days=lambda x: (x > 0).sum())
    .reset_index()
)

# Merge with the labels_df to see which ADI/CV label they originally got
all_stats = labels_df[["item_id", "label"]].merge(sales_agg_all, on="item_id", how="right")

print("Top 50 products overall by Total Sales:")
display(
    all_stats.sort_values("total_sales", ascending=False)
    .head(50)
    .reset_index(drop=True)
    .style.background_gradient(subset=["total_sales", "mean_sales"], cmap="Purples")
)